# QFT-Graph: WS1 GPU experiment runner (P1-3 / P1-4 / P1-5 / P1-6)

Runs the committed CLI experiment scripts on Colab GPU. Every number still
comes from a script in `scripts/` (plan ground rule 4) — this notebook is
only a launcher. All outputs land in `results/*.json` on Google Drive and
every script **saves incrementally**, so a disconnected runtime loses at
most the run in progress; just re-run the cell.

| Cell | Task | Output | est. GPU time |
|---|---|---|---|
| 2 | P1-4: Tables I/II seed variance @ L=16 (5 models x 5 seeds) | `results/baseline_results_v2.json` | ~2 h |
| 3 | P1-4: Table I L=64 HeteroGNN row (5 seeds) | `results/baseline_64x64_v2.json` | ~1-2 h |
| 4 | P1-5: depth ablation (3 variants x B in {1,2,3,4,6} x 3 seeds) | `results/depth_ablation.json` | ~2-4 h |
| 5 | P1-6: size transfer (2 variants x 3 seeds, eval L=8..64) | `results/size_transfer.json` | ~1-2 h |
| 6 | P1-3: multi-coupling training -> Table III' (5 seeds) | `results/table3prime.json` | ~2-3 h |

Cell 6 requires the 13-point multi-coupling dataset
(`scripts/generate_multicoupling_data.py`, running on the laptop) to have
finished syncing to Drive — the cell checks and tells you if it is not ready.

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
assert IN_COLAB, 'This runner is meant for Colab (use the CLI scripts locally)'

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
os.chdir(PROJECT_ROOT)

!pip install -q torch-geometric omegaconf h5py

# Make qft_graph importable for the subprocess script invocations below
os.environ['PYTHONPATH'] = os.path.join(PROJECT_ROOT, 'src')

import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## P1-4a — Tables I & II seed variance at L=16 (+ parameter-matched CNN)

In [ ]:
!python scripts/train_baselines.py \
    --data "data/mc_configs/phi4_16x16_m2=-0.5_lam=0.5/mc_data.pt" \
    --config configs/paper/phase1_train_16x16.yaml \
    --seeds 0 1 2 3 4 --epochs 150 --device cuda \
    --output results/baseline_results_v2.json

## P1-4b — Table I, L=64 HeteroGNN row (5 seeds)

In [ ]:
!python scripts/train_baselines.py \
    --data "data/mc_configs/phi4_64x64_m2=-0.5_lam=0.5/mc_data.pt" \
    --config configs/paper/phase1_train_64x64.yaml \
    --models HeteroGNN \
    --seeds 0 1 2 3 4 --epochs 150 --device cuda \
    --output results/baseline_64x64_v2.json

## P1-5 — Depth ablation (over-smoothing figure)

If the no-skip homogeneous variant does **not** collapse to r < 0.05, the
script flags the discrepancy vs. the manuscript's claim in its final log lines.

In [ ]:
!python scripts/run_depth_ablation.py \
    --data "data/mc_configs/phi4_16x16_m2=-0.5_lam=0.5/mc_data.pt" \
    --config configs/paper/phase1_train_16x16.yaml \
    --seeds 0 1 2 --epochs 150 --device cuda \
    --output results/depth_ablation.json

## P1-6 — Size transfer (coords vs. coordinate-free, train L=16 only)

In [ ]:
!python scripts/run_size_transfer.py \
    --seeds 0 1 2 --epochs 150 --device cuda \
    --output results/size_transfer.json

## P1-3 — Multi-coupling training -> Table III'

Trains the coupling-conditioned model jointly on alternating m^2 grid points;
holds out interleaved couplings (interpolation) and both endpoints
(extrapolation). Prints copy-paste LaTeX rows for Table III' at the end.

In [ ]:
from pathlib import Path
import numpy as np

grid = np.round(np.linspace(-2.9, -0.3, 13), 4)
missing = [f'{m2:g}' for m2 in grid
           if not Path(f'data/mc_configs/phi4_16x16_m2={m2:g}_lam=0.5/mc_data.pt').exists()]
if missing:
    print('NOT READY - multi-coupling data still generating/syncing. Missing m^2:', missing)
else:
    print('All 13 points present - launching')
    !python scripts/train_multicoupling.py --seeds 0 1 2 3 4 --epochs 150 --device cuda

## Done

Sync check: confirm the `results/*.json` files show up on the laptop's Drive
mirror, then Claude Code picks them up for the paper tables/figures
(P1-4..P1-6 part 2 commits). Do not edit the JSONs by hand.